# Run the global Rossby model

This notebook is a local validation run, not a test. It opens the supplied 1000 m isobath and ERA5–SCOTIA forcing products, coarsens only the spatial forcing grid for a quick calculation, and runs the complete temporal `GlobalRossbyModel.solve()` path. Monthly samples are treated as uniformly spaced by `365.25 / 12` days while their original calendar labels are retained on output.

In [1]:
from pathlib import Path
import sys

import numpy as np
import xarray as xr

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from moc_adjustment_theory import GlobalRossbyModel

ISOBATH_PATH = ROOT / "data/tracked/isobath/global_isobath_GEBCO_1000m.nc"
FORCING_PATH = ROOT / "data/untracked/forcing/global_ERA5_SCOTIA_forcing.nc"
MONTH_SECONDS = 365.25 / 12 * 24 * 60 * 60
G_PRIME = 0.02

In [2]:
isobath = xr.open_dataset(ISOBATH_PATH)
forcing = xr.open_dataset(FORCING_PATH)
forcing

<xarray.Dataset> Size: 1GB
Dimensions:    (time: 246, latitude: 458, longitude: 1481)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2004-01-01 2004-02-01 ... 2024-06-01
  * latitude   (latitude) float64 4kB -55.0 -54.75 -54.5 ... 58.75 59.0 59.25
  * longitude  (longitude) float64 12kB -80.0 -79.75 -79.5 ... 289.5 289.8 290.0
Data variables:
    M_Ek_x     (time, latitude, longitude) float32 667MB ...
    M_Ek_y     (time, latitude, longitude) float32 667MB ...
    T_N        (time) float64 2kB ...
Attributes: (12/13)
    title:                            Global ERA5 Ekman transport and SCOTIA ...
    source_wind_stress:               ERA5 monthly mean eastward/northward tu...
    source_northern_transport:        SCOTIA overturning diagnostics MOC
    source_geometry:                  global_isobath_GEBCO_1000m.nc
    anomaly_reference:                time mean over the common 2004-01 to 20...
    rho_0_kg_m3:                      1027.0
    ...                               ...
    active_layer_depth_m:             1000.0
    equatorial_cap_latitude_degrees:  6.876550269787968
    equatorial_gamma_s_1:             1.7461774128542137e-05
    taper_k_per_degree:               0.5
    taper_description:                smooth stress taper at solid sidewalls ...
    generated_by:                     notebooks/construct_forcing_dataset.ipynb

The source forcing grid is large. Coarsening by eight grid cells in each horizontal direction preserves the full 2004–2024 time record and all three supplied forcing variables while keeping this validation run inexpensive.

In [3]:
forcing_validation = forcing.coarsen(
    latitude=8, longitude=8, boundary="trim"
).mean()
forcing_validation

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


<xarray.Dataset> Size: 21MB
Dimensions:    (time: 246, latitude: 57, longitude: 185)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2004-01-01 2004-02-01 ... 2024-06-01
  * latitude   (latitude) float64 456B -54.12 -52.12 -50.12 ... 55.88 57.88
  * longitude  (longitude) float64 1kB -79.12 -77.12 -75.12 ... 286.9 288.9
Data variables:
    M_Ek_x     (time, latitude, longitude) float32 10MB 0.0 0.0 0.0 ... 0.0 0.0
    M_Ek_y     (time, latitude, longitude) float32 10MB 0.0 0.0 0.0 ... 0.0 0.0
    T_N        (time) float64 2kB ...
Attributes: (12/13)
    title:                            Global ERA5 Ekman transport and SCOTIA ...
    source_wind_stress:               ERA5 monthly mean eastward/northward tu...
    source_northern_transport:        SCOTIA overturning diagnostics MOC
    source_geometry:                  global_isobath_GEBCO_1000m.nc
    anomaly_reference:                time mean over the common 2004-01 to 20...
    rho_0_kg_m3:                      1027.0
    ...                               ...
    active_layer_depth_m:             1000.0
    equatorial_cap_latitude_degrees:  6.876550269787968
    equatorial_gamma_s_1:             1.7461774128542137e-05
    taper_k_per_degree:               0.5
    taper_description:                smooth stress taper at solid sidewalls ...
    generated_by:                     notebooks/construct_forcing_dataset.ipynb

In [4]:
model = GlobalRossbyModel(isobath, g_prime=G_PRIME)
solution = model.solve(
    forcing_validation,
    sample_spacing_seconds=MONTH_SECONDS,
)
solution

<xarray.Dataset> Size: 3MB
Dimensions:   (region: 5, time: 246, latitude: 57)
Coordinates:
  * region    (region) <U16 320B 'north_atlantic' ... 'atlantic_pacific'
  * time      (time) datetime64[ns] 2kB 2004-01-01 2004-02-01 ... 2024-06-01
  * latitude  (latitude) float64 456B -54.12 -52.12 -50.12 ... 53.88 55.88 57.88
Data variables:
    h_e       (time, region) float64 10kB -0.5117 0.75 -0.2765 ... 1.426 -2.665
    h_b       (time, region, latitude) float64 561kB nan nan nan ... nan nan nan
    h_w       (time, region, latitude) float64 561kB nan nan nan ... nan nan nan
    T         (time, region, latitude) float64 561kB nan nan nan ... nan nan nan
    T_g       (time, region, latitude) float64 561kB nan nan nan ... nan nan nan
    T_Ek      (time, region, latitude) float64 561kB nan nan nan ... nan nan nan
Attributes:
    g_prime_m_s-2:    0.02
    isobath_depth_m:  1000.0

In [5]:
validation = {
    "calendar_labels_restored": bool(np.array_equal(solution.time, forcing_validation.time)),
    "all_outputs_have_finite_active_values": {
        name: bool(np.isfinite(array.values).any())
        for name, array in solution.data_vars.items()
    },
    "max_transport_closure_m3_s-1": float(
        abs(solution.T - solution.T_g - solution.T_Ek).max(skipna=True)
    ),
}
validation

{'calendar_labels_restored': True,
 'all_outputs_have_finite_active_values': {'h_e': True,
  'h_b': True,
  'h_w': True,
  'T': True,
  'T_g': True,
  'T_Ek': True},
 'max_transport_closure_m3_s-1': 1.1175870895385742e-08}